In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt

In [ ]:
# 0. 基本設定
# =========================
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 256
NUM_CLASSES = 16

TRAIN_FOLDER = "final_subclass_split/train"
TEST_FOLDER = "final_subclass_split/test"
CSV_DIR = "datasets"
WEIGHTS_PATH = "best_hybrid_model.weights.h5"   # 你的權重檔
FEATURE_LAYER_NAME = "cnn_demse_64"             # 建議先用這層當 CNN feature

In [3]:
def prepare_image_and_numeric_data(subset_name):
    """
    讀取:
    - 圖片路徑
    - label
    - period
    - windex

    並確保順序完全對齊
    """
    base_path = f"final_subclass_split/{subset_name}"
    csv_dir = "datasets"

    csv_files = [
        f for f in os.listdir(csv_dir)
        if f.startswith(subset_name) and f.endswith(".csv")
    ]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found for subset '{subset_name}' in {csv_dir}")

    df_list = [pd.read_csv(os.path.join(csv_dir, f)) for f in csv_files]
    master_df = pd.concat(df_list, ignore_index=True).set_index("id")

    # 跟原本 hybrid model 一樣：缺 period / windex 的資料不要
    master_df = master_df.dropna(subset=["period", "windex"])

    file_paths = []
    labels = []
    periods = []
    windexes = []

    class_names = sorted([
        name for name in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, name))
    ])
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    print(f"Syncing {subset_name} data...")

    for class_name in class_names:
        class_dir = os.path.join(base_path, class_name)

        for img_name in os.listdir(class_dir):
            if not img_name.endswith(".png"):
                continue

            star_id = img_name.replace(".png", "")
            if star_id in master_df.index:
                file_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_to_idx[class_name])
                periods.append(master_df.loc[star_id, "period"])
                windexes.append(master_df.loc[star_id, "windex"])

    file_paths = np.array(file_paths)
    labels = np.array(labels, dtype=np.int32)
    periods = np.array(periods, dtype=np.float32).reshape(-1, 1)
    windexes = np.array(windexes, dtype=np.float32).reshape(-1, 1)

    print(f"{subset_name}: {len(file_paths)} samples")
    print(f"{subset_name} period shape: {periods.shape}")
    print(f"{subset_name} windex shape: {windexes.shape}")

    return file_paths, labels, periods, windexes, class_names

In [4]:
# Image-only tf.data dataset
# =========================
def load_image_only(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def make_image_dataset(file_paths, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(load_image_only, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(file_paths), seed=123)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [5]:
# 2. 重建原本 hybrid model
# =========================
def build_variable_star_cnn(input_shape=(128, 128, 1), num_classes=16, dropout_rate=0.3):
    image_input = layers.Input(shape=input_shape, name="image_input")

    # Block 1
    x = layers.Conv2D(32, (7, 7), activation="relu", padding="valid", name="conv1_7x7")(image_input)
    x = layers.Conv2D(32, (5, 5), activation="relu", padding="valid", name="conv2_5x5")(x)
    x = layers.Dropout(dropout_rate, name="dropout1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="valid", name="conv3_3x3")(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="valid", name="conv4_3x3")(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="valid", name="conv5_3x3")(x)
    x = layers.Dropout(dropout_rate, name="dropout2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="valid", name="conv6_3x3")(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="valid", name="conv7_3x3")(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="valid", name="conv8_3x3")(x)
    x = layers.Dropout(dropout_rate, name="dropout3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), activation="relu", padding="valid", name="conv9_3x3")(x)
    x = layers.Conv2D(256, (3, 3), activation="relu", padding="valid", name="conv10_3x3")(x)
    x = layers.Dropout(dropout_rate, name="dropout4")(x)
    x = layers.MaxPooling2D((2, 2), name="pool4")(x)

    # Image branch FC
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(256, activation="relu", name="cnn_dense_256")(x)
    x = layers.Dense(64, activation="relu", name="cnn_dense_64")(x)
    cnn_features = layers.Dense(16, activation="softmax", name="cnn_dense_16")(x)

    # Period branch
    period_input = layers.Input(shape=(1,), name="period_input")
    period_features = layers.Dense(16, activation="softmax", name="period_dense_16")(period_input)

    # W-index branch
    w_input = layers.Input(shape=(1,), name="w_input")
    w_features = layers.Dense(16, activation="softmax", name="w_dense_16")(w_input)

    # Merge
    merged = layers.concatenate([cnn_features, period_features, w_features], name="concat")
    x = layers.Dense(64, activation="relu", name="merged_dense_64")(merged)
    output = layers.Dense(num_classes, activation="softmax", name="final_output")(x)

    return models.Model(inputs=[image_input, period_input, w_input], outputs=output)

In [6]:
# 4. 建立 feature extractor
# =========================
def build_feature_extractor(weights_path, feature_layer_name="flatten"):
    """
    先重建完整 hybrid model，再載入 weights，
    最後只保留 image_input -> 指定 CNN layer 的輸出
    """
    full_model = build_variable_star_cnn(num_classes=16)
    full_model.load_weights(weights_path)
    print(f"Loaded weights from: {weights_path}")

    feature_extractor = Model(
        inputs=full_model.input[0],   # image_input
        outputs=full_model.get_layer(feature_layer_name).output
    )

    print("Feature extractor built successfully.")
    print("Feature layer:", feature_layer_name)
    return feature_extractor

In [7]:
# 5. Extract CNN features
# =========================
def extract_features(feature_extractor, dataset):
    features = []
    labels = []

    for batch_imgs, batch_labels in dataset:
        batch_feat = feature_extractor.predict(batch_imgs, verbose=0)
        features.append(batch_feat)
        labels.append(batch_labels.numpy())

    X = np.concatenate(features, axis=0)
    y = np.concatenate(labels, axis=0)
    return X, y

In [8]:
# 6. Plot confusion matrix
# =========================
def plot_confusion_matrix(cm_norm, class_names, save_path="svm_confusion_matrix.png"):
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.set_title("SVM on CNN Features + Numerical Features")

    thresh = 0.5
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i, j]
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if val > thresh else "black"
            )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    print(f"Saved confusion matrix -> {save_path}")

In [9]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [ ]:
# 7. Main
# =========================
def main():
    # ---- Load train / test data
    train_paths, y_train_raw, period_train, w_train, class_names = prepare_image_and_numeric_data("train")
    test_paths, y_test_raw, period_test, w_test, class_names_test = prepare_image_and_numeric_data("test")

    if class_names != class_names_test:
        raise ValueError("Train/Test class_names do not match.")

    # ---- Build image datasets
    train_ds = make_image_dataset(train_paths, y_train_raw, shuffle=False)
    test_ds = make_image_dataset(test_paths, y_test_raw, shuffle=False)

    # ---- Build feature extractor
    feature_extractor = build_feature_extractor(
        weights_path=WEIGHTS_PATH,
        feature_layer_name=FEATURE_LAYER_NAME
    )

    # ---- Extract CNN features
    print("Extracting train image features...")
    X_train_feat, y_train = extract_features(feature_extractor, train_ds)

    print("Extracting test image features...")
    X_test_feat, y_test = extract_features(feature_extractor, test_ds)

    print("X_train_feat shape:", X_train_feat.shape)
    print("X_test_feat shape:", X_test_feat.shape)

    # ---- Concatenate image features + numerical features
    X_train_final = np.concatenate([X_train_feat, period_train, w_train], axis=1)
    X_test_final = np.concatenate([X_test_feat, period_test, w_test], axis=1)

    print("X_train_final shape:", X_train_final.shape)
    print("X_test_final shape:", X_test_final.shape)

    # ---- Standardize
    scaler_cnn = StandardScaler()
    scaler_num = StandardScaler()

    X_train_cnn = scaler_cnn.fit_transform(X_train_feat)
    X_test_cnn = scaler_cnn.transform(X_test_feat)

    from sklearn.decomposition import PCA

    pca = PCA(n_components=0.95)

    X_train_cnn = pca.fit_transform(X_train_cnn)
    X_test_cnn = pca.transform(X_test_cnn)

    X_train_num = scaler_num.fit_transform(np.concatenate([period_train, w_train], axis=1))
    X_test_num = scaler_num.transform(np.concatenate([period_test, w_test], axis=1))

    X_train_scaled = np.concatenate([X_train_cnn, X_train_num], axis=1)
    X_test_scaled = np.concatenate([X_test_cnn, X_test_num], axis=1)

    # ---- Grid Search for SVM
    print("Running Grid Search for SVM...")

    param_grid = {
        "C": [0.5, 1, 5, 10],
        "gamma": ["scale", 0.01],
        "kernel": ["rbf"]
    }

    grid = GridSearchCV(
        estimator=SVC(class_weight="balanced"),
        param_grid=param_grid,
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    grid.fit(X_train_scaled, y_train)
    svm_clf = grid.best_estimator_

    # ---- Predict
    y_pred = svm_clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {acc:.4f}")

    # ---- Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1_total = f1_score(y_test, y_pred, average='weighted')

    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test F1-score (Weighted): {f1_total:.4f}")

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    plot_confusion_matrix(cm_norm, class_names, save_path="svm_confusion_matrix.png")

    # ---- Save results
    results = {
        "feature_layer": FEATURE_LAYER_NAME,
        "accuracy": float(acc),
        "f1_score": float(f1_total),
        "best_params": grid.best_params_,
        "best_cv_score": float(grid.best_score_),
        "class_names": class_names,
        "confusion_matrix": cm.tolist(),
        "confusion_matrix_norm": cm_norm.tolist(),
        "train_feature_shape": list(X_train_feat.shape),
        "test_feature_shape": list(X_test_feat.shape),
        "train_final_shape": list(X_train_final.shape),
        "test_final_shape": list(X_test_final.shape),
    }

    with open("svm_results_with_numeric.json", "w") as f:
        json.dump(results, f, indent=2)

    print("Saved results -> svm_results_with_numeric.json")


if __name__ == "__main__":
    main()

Syncing train data...
train: 71201 samples
train period shape: (71201, 1)
train windex shape: (71201, 1)
Syncing test data...
test: 7948 samples
test period shape: (7948, 1)
test windex shape: (7948, 1)
Loaded weights from: best_hybrid_model.weights.h5
Feature extractor built successfully.
Feature layer: flatten
Extracting train image features...
Extracting test image features...
X_train_feat shape: (71201, 2304)
X_test_feat shape: (7948, 2304)
X_train_final shape: (71201, 2306)
X_test_final shape: (7948, 2306)
Running Grid Search for SVM...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
